#### In this notebook we build a frame containing
* A 4-level index: INDEX 1 = dataset/model ; INDEX 2 = $\kappa_f$ ; INDEX 3 = parameter (n, $\delta$, imbalance); INDEX 4 = parameters value 
* For each index, for each metric, we compute the 50 seeds-Average Distance Between Curves (noted ADBC, _between curves_ meaning between _bounds_ curve and _test metric_ curve)

In [1]:
%run -i ../python_scripts/nb_setup.py

In [2]:
datasets_paths = [
    "../experiments/CIFAR/sgp_set_cnn",
    "../experiments/CIFAR/sgp_set_cnn_MCD",
    "../experiments/WSI/sgp_set_cnn",
    "../experiments/WSI/sgp_set_cnn_MCD",
]

In [3]:
num_seeds = 500

In [4]:
dicos = []
datasets = ["CIFAR", "WSI"]
models = ["cnn"]

for path in datasets_paths:

    print("Considering ", path)
    mod = models[int(np.where([(m in path) for m in models])[0][0])]
    ds = datasets[int(np.where([(d in path) for d in datasets])[0][0])]
    if "MCD" in path:
        kappa = "MCD"
    else:
        kappa = "SR"

    original_ds = pickle.load(open(path, "rb"))

    print(r"$\delta$ values...")
    for delta in [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]:
        print("delta = ", delta)
        for metric in ["standard", "FP", "FN", "FPR", "FNR"]:
            print("metric = ", metric)
            values = []
            for seed in tqdm(range(num_seeds)):
                df = original_ds.sample(frac=1, random_state=seed)
                if kappa == "MCD":
                    values.append(
                        ABC(
                            df, metric=metric, delta=delta, theta_min=-0.05, theta_max=0
                        )
                    )
                else:  # kappa == 'SR'
                    values.append(ABC(df, metric=metric, delta=delta))
            dicos.append(
                {
                    "dataset": ds,
                    "model": mod,
                    "kappa": kappa,
                    "metric": metric,
                    "param.": "delta",
                    "value": delta,
                    "ADBC": np.mean(values),
                    "CI": [
                        np.mean(values) - 1.96 * np.std(values),
                        np.mean(values) + 1.96 * np.std(values),
                    ],
                }
            )

    print("Sample size proportions...")
    for prop in np.linspace(1 / 4, 1, num=6):
        print("prop = ", prop)
        for metric in ["standard", "FP", "FN", "FPR", "FNR"]:
            print("metric = ", metric)
            values = []
            for seed in tqdm(range(num_seeds)):
                df = original_ds.sample(frac=prop, random_state=seed)
                if kappa == "MCD":
                    values.append(ABC(df, metric=metric, theta_min=-0.05, theta_max=0))
                else:  # kappa == 'SR'
                    values.append(ABC(df, metric=metric))
            dicos.append(
                {
                    "dataset": ds,
                    "model": mod,
                    "kappa": kappa,
                    "metric": metric,
                    "param.": "N",
                    "value": prop,
                    "ADBC": np.mean(values),
                    "CI": [
                        np.mean(values) - 1.96 * np.std(values),
                        np.mean(values) + 1.96 * np.std(values),
                    ],
                }
            )

    print("Imbalance rates...")
    for imbalance in np.linspace(1 / 6, 5 / 6, num=5):
        print("imbalance = ", imbalance)
        for metric in ["standard", "FP", "FN", "FPR", "FNR"]:
            print("metric = ", metric)
            values = []
            for seed in tqdm(range(num_seeds)):
                df = generate_imbalanced_datasets(
                    original_ds, [imbalance], seed=seed, fixed=True
                )[0]
                if kappa == "MCD":
                    values.append(ABC(df, metric=metric, theta_min=-0.05, theta_max=0))
                else:  # kappa == 'SR'
                    values.append(ABC(df, metric=metric))
            dicos.append(
                {
                    "dataset": ds,
                    "model": mod,
                    "kappa": kappa,
                    "metric": metric,
                    "param.": "imbalance",
                    "value": imbalance,
                    "ADBC": np.mean(values),
                    "CI": [
                        np.mean(values) - 1.96 * np.std(values),
                        np.mean(values) + 1.96 * np.std(values),
                    ],
                }
            )

    clear_output(wait=True)

Considering  ../experiments/WSI/sgp_set_cnn_MCD
$\delta$ values...
delta =  1e-05
metric =  standard


100%|██████████| 500/500 [00:25<00:00, 19.64it/s]


metric =  FP


100%|██████████| 500/500 [00:33<00:00, 14.90it/s]


metric =  FN


...

100%|██████████| 500/500 [00:34<00:00, 14.35it/s]


metric =  FPR


100%|██████████| 500/500 [00:53<00:00,  9.42it/s]


metric =  FNR


100%|██████████| 500/500 [00:55<00:00,  8.99it/s]


Displaying resulting frame

In [5]:
res = pd.DataFrame(dicos)
display(res)

,dataset,model,kappa,metric,param.,value,ADBC,CI
0,CIFAR,cnn,SR,standard,delta,0.000010,0.029689,"[0.023500557941836052, 0.035878381272653824]"
1,CIFAR,cnn,SR,FP,delta,0.000010,0.028847,"[0.02310244800079632, 0.034591684965528016]"
2,CIFAR,cnn,SR,FN,delta,0.000010,0.021146,"[0.01849905999941329, 0.023793181788471045]"
3,CIFAR,cnn,SR,FPR,delta,0.000010,0.031166,"[0.02475202151322911, 0.03757987912130612]"
4,CIFAR,cnn,SR,FNR,delta,0.000010,0.089841,"[0.06764800363389098, 0.11203358297665442]"
...,...,...,...,...,...,...,...,...
315,WSI,cnn,MCD,standard,imbalance,0.833333,0.014298,"[0.005049440752533498, 0.02354729413112721]"
316,WSI,cnn,MCD,FP,imbalance,0.833333,0.010394,"[0.0037417333456130526, 0.01704542236437434]"
317,WSI,cnn,MCD,FN,imbalance,0.833333,0.011206,"[0.004206411274791063, 0.018204966044936745]"
318,WSI,cnn,MCD,FPR,imbalance,0.833333,0.058200,"[0.01984782601301866, 0.09655274241358794]"


Displaying the evolution of ADBC with respect to parameters

In [6]:
# Define custom order
order = ["standard", "FP", "FN", "FPR", "FNR"]
res["metric"] = pd.Categorical(res["metric"], categories=order, ordered=True)

In [7]:
mpl.rcParams["figure.dpi"] = 300
if isinstance(res["CI"].iloc[0], str):
    res["CI"] = res["CI"].apply(ast.literal_eval)

# Split CI into bounds
res["CI_low"] = res["CI"].apply(lambda x: x[0])
res["CI_high"] = res["CI"].apply(lambda x: x[1])

# --- layout: rows = (dataset, model, metric), cols = param. ---
row_groups = list(res.groupby(["dataset", "model", "metric"]))
n_rows = len(row_groups)


def n_params(g):
    return g["param."].nunique()


n_cols = max(n_params(g) for _, g in row_groups)

rc = {
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "axes.linewidth": 1.0,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.frameon": True,
    "legend.framealpha": 0.9,
    "legend.facecolor": "white",
    "legend.edgecolor": "#D0D0D0",
    "grid.alpha": 0.45,
    "lines.linewidth": 2.0,
}
with plt.rc_context(rc), plt.style.context("seaborn-v0_8-whitegrid"):
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), squeeze=False
    )
    fig.patch.set_facecolor("white")

    for row_idx, ((dataset, model, metric), g_row) in enumerate(row_groups):
        params_order = list(dict.fromkeys(g_row["param."].tolist()))
        if len(params_order) == 0:
            continue

        for col_idx, param in enumerate(params_order):
            ax = axes[row_idx, col_idx]
            g_param = g_row[g_row["param."] == param]

            # style
            for spine in ax.spines.values():
                spine.set_color("#B0B0B0")
                spine.set_linewidth(0.9)
            ax.xaxis.set_minor_locator(AutoMinorLocator())
            ax.yaxis.set_minor_locator(AutoMinorLocator())
            ax.grid(True, which="major", linestyle="--", linewidth=0.7, alpha=0.5)
            ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.3)

            # one line per kappa
            for kappa, g_k in g_param.groupby("kappa"):
                g_k = g_k.sort_values("value")
                (line,) = ax.plot(
                    g_k["value"],
                    g_k["ADBC"],
                    marker="o",
                    markersize=4,
                    linewidth=2,
                    label=f"{kappa}",
                )
                ax.fill_between(
                    g_k["value"],
                    g_k["CI_low"],
                    g_k["CI_high"],
                    alpha=0.18,
                    color=line.get_color(),
                )

            # titles + labels
            ax.set_title(f"{dataset} • {model}")
            if param == "delta":
                ax.set_xlabel(r"$\delta$")
                ax.set_xscale("log")  # log scale for delta
            else:
                ax.set_xlabel(param)
            # if col_idx == 0:
            ax.set_ylabel(f"{metric} ADBC")

            ax.legend(title="kappa", fontsize=9, title_fontsize=10, loc="best")

        for col_idx in range(len(params_order), n_cols):
            fig.delaxes(axes[row_idx, col_idx])

    plt.tight_layout()
    plt.show()